# Week 8-9 — Transfer Learning: ResNet50, MobileNetV2, EfficientNet-B0

Notebook này tiếp nối `week67_alexnet-from-scratch.ipynb` và tập trung vào:

- Áp dụng **transfer learning** với 3 backbone pretrained trên ImageNet:
  - `ResNet50` — baseline phổ biến, 25M params
  - `MobileNetV2` — lightweight, tối ưu cho mobile/edge
  - `EfficientNet-B0` — cân bằng accuracy/FLOPs tốt nhất (compound scaling)
- Chiến lược **2 pha**:
  - **Phase 1 — Feature Extraction**: freeze toàn bộ backbone, chỉ train classifier head mới
  - **Phase 2 — Fine-tuning**: unfreeze backbone, train end-to-end với differential learning rate
- So sánh kết quả với AlexNet from scratch (tuần 6-7)

Nguồn tham khảo:
- [torchvision models](https://pytorch.org/vision/stable/models.html)
- *How transferable are features in deep neural networks?* (Yosinski et al., 2014)
- *EfficientNet: Rethinking Model Scaling for CNNs* (Tan & Le, 2019)

## Chuẩn bị môi trường

Notebook này giả định bạn đã chạy notebook tuần 5 để sinh ra:

- `artifacts/datasets/class_to_idx.json`
- `artifacts/datasets/train_records.json`
- `artifacts/datasets/val_records.json`
- `artifacts/datasets/test_records.json`

Transfer learning dùng ImageNet pretrained weights nên **bắt buộc** dùng ImageNet normalization (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`).

In [ ]:
from __future__ import annotations

import json
import random
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode

In [ ]:
ARTIFACTS_DIR = Path("./artifacts/datasets")
CHECKPOINT_DIR = Path("./artifacts/checkpoints")
HISTORY_DIR = Path("./artifacts/training")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

CLASS_TO_IDX_PATH = ARTIFACTS_DIR / "class_to_idx.json"
TRAIN_RECORDS_PATH = ARTIFACTS_DIR / "train_records.json"
VAL_RECORDS_PATH = ARTIFACTS_DIR / "val_records.json"
TEST_RECORDS_PATH = ARTIFACTS_DIR / "test_records.json"

IMAGE_SIZE = 224
RESIZE_SIZE = 256

BATCH_SIZE = 32
NUM_WORKERS = 8

PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 15

PHASE1_LR = 1e-3
PHASE2_LR_BACKBONE = 1e-5
PHASE2_LR_HEAD = 1e-4
WEIGHT_DECAY = 1e-4

USE_BBOX = False
BBOX_PADDING = 0.05

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print(f"Device          : {DEVICE}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Num workers     : {NUM_WORKERS}")
print(f"Phase 1 epochs  : {PHASE1_EPOCHS}")
print(f"Phase 2 epochs  : {PHASE2_EPOCHS}")

In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file: {path}. Hãy chạy notebook tuần 5 trước."
        )
    return json.loads(path.read_text(encoding="utf-8"))


def parse_annotation(annotation_path: str) -> dict:
    root = ET.parse(annotation_path).getroot()
    size_node = root.find("size")
    object_node = root.find("object")
    bbox_node = object_node.find("bndbox")

    return {
        "width": int(size_node.findtext("width")),
        "height": int(size_node.findtext("height")),
        "bbox": {
            "xmin": int(bbox_node.findtext("xmin")),
            "ymin": int(bbox_node.findtext("ymin")),
            "xmax": int(bbox_node.findtext("xmax")),
            "ymax": int(bbox_node.findtext("ymax")),
        },
    }


def crop_with_bbox(
    image: Image.Image,
    bbox: dict,
    padding_ratio: float = 0.0,
) -> Image.Image:
    xmin, ymin, xmax, ymax = bbox["xmin"], bbox["ymin"], bbox["xmax"], bbox["ymax"]
    box_width = xmax - xmin
    box_height = ymax - ymin

    pad_x = int(box_width * padding_ratio)
    pad_y = int(box_height * padding_ratio)

    xmin = max(0, xmin - pad_x)
    ymin = max(0, ymin - pad_y)
    xmax = min(image.width, xmax + pad_x)
    ymax = min(image.height, ymax + pad_y)

    return image.crop((xmin, ymin, xmax, ymax))


def accuracy_at_k(logits: torch.Tensor, targets: torch.Tensor, topk=(1, 5)) -> dict[int, float]:
    max_k = min(max(topk), logits.shape[1])
    _, pred = logits.topk(max_k, dim=1, largest=True, sorted=True)
    pred = pred.t()
    correct = pred.eq(targets.view(1, -1).expand_as(pred))

    results = {}
    batch_size = targets.size(0)
    for k in topk:
        k = min(k, logits.shape[1])
        correct_k = correct[:k].reshape(-1).float().sum(0)
        results[k] = (correct_k / batch_size).item()
    return results


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_total_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


seed_everything(RANDOM_SEED)

In [ ]:
class_to_idx = load_json(CLASS_TO_IDX_PATH)
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

train_records = load_json(TRAIN_RECORDS_PATH)
val_records = load_json(VAL_RECORDS_PATH)
test_records = load_json(TEST_RECORDS_PATH)

num_classes = len(class_to_idx)

print(f"Num classes   : {num_classes}")
print(f"Train records : {len(train_records)}")
print(f"Val records   : {len(val_records)}")
print(f"Test records  : {len(test_records)}")
print(json.dumps(train_records[0], indent=2, ensure_ascii=False))

## Dataset và transforms

Transfer learning dùng pretrained weights nên **bắt buộc** sử dụng ImageNet normalization.

Data augmentation pipeline:
- **Train**: `Resize(256)` → `RandomResizedCrop(224)` → `RandomHorizontalFlip` → `ColorJitter` → `Normalize`
- **Val/Test**: `Resize(256)` → `CenterCrop(224)` → `Normalize`

So với AlexNet notebook, ta thêm `RandomResizedCrop` và `ColorJitter` nhẹ — hai augmentation phổ biến khi fine-tune pretrained models giúp tránh overfit trên dataset nhỏ.

In [ ]:
class StanfordDogsDataset(Dataset):
    def __init__(
        self,
        records: list[dict],
        transform=None,
        use_bbox: bool = False,
        bbox_padding: float = 0.0,
    ) -> None:
        self.records = list(records)
        self.transform = transform
        self.use_bbox = use_bbox
        self.bbox_padding = bbox_padding

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        record = self.records[index]

        with Image.open(record["image_path"]) as pil_image:
            image = pil_image.convert("RGB")

        if self.use_bbox:
            annotation = parse_annotation(record["annotation_path"])
            image = crop_with_bbox(
                image,
                annotation["bbox"],
                padding_ratio=self.bbox_padding,
            )

        if self.transform is not None:
            image = self.transform(image)

        label = record["class_id"]
        return image, label


train_transform = transforms.Compose(
    [
        transforms.Resize(RESIZE_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize(RESIZE_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

train_dataset = StanfordDogsDataset(
    train_records,
    transform=train_transform,
    use_bbox=USE_BBOX,
    bbox_padding=BBOX_PADDING,
)

val_dataset = StanfordDogsDataset(
    val_records,
    transform=eval_transform,
    use_bbox=USE_BBOX,
    bbox_padding=BBOX_PADDING,
)

test_dataset = StanfordDogsDataset(
    test_records,
    transform=eval_transform,
    use_bbox=USE_BBOX,
    bbox_padding=BBOX_PADDING,
)

DATALOADER_KWARGS = {
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
}
if NUM_WORKERS > 0:
    DATALOADER_KWARGS["persistent_workers"] = True
    DATALOADER_KWARGS["prefetch_factor"] = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    **DATALOADER_KWARGS,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **DATALOADER_KWARGS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **DATALOADER_KWARGS,
)

images, labels = next(iter(train_loader))
print(f"Train batch images shape: {tuple(images.shape)}")
print(f"Train batch labels shape: {tuple(labels.shape)}")

## Model factory

Hàm `build_transfer_model` nhận tên backbone và trả về model đã:
1. Load pretrained ImageNet weights
2. Thay classifier head cuối cùng cho 120 classes
3. Freeze toàn bộ backbone (cho Phase 1 — feature extraction)

Mỗi backbone có cách thay head khác nhau:
- **ResNet50**: thay `model.fc` (Linear)
- **MobileNetV2**: thay `model.classifier[1]` (Linear)
- **EfficientNet-B0**: thay `model.classifier[1]` (Linear)

In [ ]:
BACKBONE_REGISTRY: dict[str, dict] = {
    "resnet50": {
        "factory": lambda: models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2),
        "head_attr": "fc",
        "feature_dim": 2048,
    },
    "mobilenet_v2": {
        "factory": lambda: models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2),
        "head_attr": "classifier.1",
        "feature_dim": 1280,
    },
    "efficientnet_b0": {
        "factory": lambda: models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1),
        "head_attr": "classifier.1",
        "feature_dim": 1280,
    },
}


def _set_nested_attr(model: nn.Module, attr_path: str, value: nn.Module) -> None:
    """Set a nested attribute like 'classifier.1' on a module."""
    parts = attr_path.split(".")
    parent = model
    for part in parts[:-1]:
        parent = getattr(parent, part) if not part.isdigit() else parent[int(part)]
    last = parts[-1]
    if last.isdigit():
        parent[int(last)] = value
    else:
        setattr(parent, last, value)


def _get_nested_attr(model: nn.Module, attr_path: str) -> nn.Module:
    obj = model
    for part in attr_path.split("."):
        obj = getattr(obj, part) if not part.isdigit() else obj[int(part)]
    return obj


def build_transfer_model(
    backbone_name: str,
    num_classes: int,
    dropout: float = 0.3,
    freeze_backbone: bool = True,
) -> nn.Module:
    if backbone_name not in BACKBONE_REGISTRY:
        raise ValueError(f"Unknown backbone: {backbone_name}. Choose from {list(BACKBONE_REGISTRY)}")

    config = BACKBONE_REGISTRY[backbone_name]
    model = config["factory"]()
    feature_dim = config["feature_dim"]

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    new_head = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(feature_dim, num_classes),
    )

    if backbone_name == "resnet50":
        model.fc = new_head
    else:
        model.classifier = new_head

    return model


def unfreeze_backbone(model: nn.Module) -> None:
    for param in model.parameters():
        param.requires_grad = True


def get_param_groups(
    model: nn.Module,
    backbone_name: str,
    lr_backbone: float,
    lr_head: float,
) -> list[dict]:
    """Split parameters into backbone vs head groups with different LRs."""
    if backbone_name == "resnet50":
        head_params = set(id(p) for p in model.fc.parameters())
    else:
        head_params = set(id(p) for p in model.classifier.parameters())

    backbone_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_params]
    head_params_list = [p for p in model.parameters() if p.requires_grad and id(p) in head_params]

    return [
        {"params": backbone_params, "lr": lr_backbone},
        {"params": head_params_list, "lr": lr_head},
    ]


for name in BACKBONE_REGISTRY:
    m = build_transfer_model(name, num_classes=num_classes, freeze_backbone=True)
    m = m.to(DEVICE)
    dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    out = m(dummy)
    print(
        f"{name:20s} | output: {tuple(out.shape)} | "
        f"total params: {count_total_parameters(m):>12,} | "
        f"trainable (Phase 1): {count_trainable_parameters(m):>10,}"
    )
    del m, dummy, out
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Training engine

Reuse training loop từ tuần 6-7 với một số cải tiến cho transfer learning:

- **Phase 1 (Feature Extraction)**: freeze backbone, dùng `AdamW` chỉ train head, LR `1e-3`
  - **BatchNorm freeze**: ngoài `requires_grad=False`, ta còn giữ tất cả BN layers ở `eval()` mode
    trong Phase 1 để running mean/var không bị cập nhật bởi dữ liệu mới (bảo toàn pretrained statistics)
- **Phase 2 (Fine-tuning)**: unfreeze backbone, dùng `AdamW` với **differential LR**:
  - Backbone: `1e-5` (tránh phá pretrained features)
  - Head: `1e-4`
  - BN layers trở lại `train()` mode bình thường để adapt vào domain mới
- Scheduler: `CosineAnnealingLR` — phổ biến hơn `ReduceLROnPlateau` cho fine-tuning
- Label smoothing `0.1` — giảm overconfidence trên dataset nhỏ

In [ ]:
def _freeze_bn(model: nn.Module) -> None:
    """Set all BatchNorm layers to eval mode (preserves pretrained running stats)."""
    for module in model.modules():
        if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d, nn.SyncBatchNorm)):
            module.eval()


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    scaler: torch.amp.GradScaler,
    use_amp: bool = False,
    freeze_bn: bool = False,
) -> dict:
    model.train()
    if freeze_bn:
        _freeze_bn(model)

    running_loss = 0.0
    running_top1 = 0.0
    running_top5 = 0.0
    seen_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        metrics = accuracy_at_k(logits.detach(), labels, topk=(1, 5))

        running_loss += loss.item() * batch_size
        running_top1 += metrics[1] * batch_size
        running_top5 += metrics[5] * batch_size
        seen_samples += batch_size

    return {
        "loss": running_loss / seen_samples,
        "top1": running_top1 / seen_samples,
        "top5": running_top5 / seen_samples,
    }


@torch.inference_mode()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict:
    model.eval()

    running_loss = 0.0
    running_top1 = 0.0
    running_top5 = 0.0
    seen_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        metrics = accuracy_at_k(logits, labels, topk=(1, 5))

        running_loss += loss.item() * batch_size
        running_top1 += metrics[1] * batch_size
        running_top5 += metrics[5] * batch_size
        seen_samples += batch_size

    return {
        "loss": running_loss / seen_samples,
        "top1": running_top1 / seen_samples,
        "top5": running_top5 / seen_samples,
    }

In [ ]:
def run_transfer_pipeline(
    backbone_name: str,
    num_classes: int,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    use_amp: bool,
    phase1_epochs: int,
    phase2_epochs: int,
    phase1_lr: float,
    phase2_lr_backbone: float,
    phase2_lr_head: float,
    weight_decay: float,
    checkpoint_dir: Path,
    history_dir: Path,
) -> tuple[nn.Module, list[dict]]:
    """Two-phase transfer learning: feature extraction then fine-tuning."""

    tag = backbone_name
    checkpoint_path = checkpoint_dir / f"{tag}_best.pt"
    history_path = history_dir / f"{tag}_history.json"

    print(f"\n{'='*70}")
    print(f"  BACKBONE: {backbone_name.upper()}")
    print(f"{'='*70}")

    # ── Phase 1: Feature Extraction ─────────────────────────────────────
    print(f"\n--- Phase 1: Feature Extraction ({phase1_epochs} epochs) ---")

    seed_everything(RANDOM_SEED)
    model = build_transfer_model(backbone_name, num_classes, dropout=0.3, freeze_backbone=True)
    model = model.to(device)

    print(f"Trainable params: {count_trainable_parameters(model):,} / {count_total_parameters(model):,}")

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=phase1_lr,
        weight_decay=weight_decay,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=phase1_epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    history: list[dict] = []
    best_val_top1 = -float("inf")

    for epoch in range(1, phase1_epochs + 1):
        start_time = time.time()

        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler, use_amp, freeze_bn=True)
        val_metrics = evaluate(model, val_loader, criterion, device)

        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]
        epoch_time = time.time() - start_time

        record = {
            "phase": 1,
            "epoch": epoch,
            "lr": current_lr,
            "train_loss": train_metrics["loss"],
            "train_top1": train_metrics["top1"],
            "train_top5": train_metrics["top5"],
            "val_loss": val_metrics["loss"],
            "val_top1": val_metrics["top1"],
            "val_top5": val_metrics["top5"],
            "epoch_time_sec": epoch_time,
        }
        history.append(record)

        marker = ""
        if record["val_top1"] > best_val_top1:
            best_val_top1 = record["val_top1"]
            torch.save(
                {
                    "epoch": epoch,
                    "phase": 1,
                    "backbone": backbone_name,
                    "model_state_dict": model.state_dict(),
                    "best_val_top1": best_val_top1,
                    "history": history,
                    "class_to_idx": class_to_idx,
                },
                checkpoint_path,
            )
            marker = " ★"

        print(
            f"  P1 Epoch {epoch:02d}/{phase1_epochs} | "
            f"lr={current_lr:.6f} | "
            f"train_top1={record['train_top1']:.4f} | "
            f"val_top1={record['val_top1']:.4f} | val_top5={record['val_top5']:.4f} | "
            f"{epoch_time:.1f}s{marker}"
        )

    print(f"  Phase 1 best val_top1: {best_val_top1:.4f}")

    # ── Phase 2: Fine-tuning ────────────────────────────────────────────
    print(f"\n--- Phase 2: Fine-tuning ({phase2_epochs} epochs) ---")

    unfreeze_backbone(model)
    print(f"Trainable params: {count_trainable_parameters(model):,} / {count_total_parameters(model):,}")

    param_groups = get_param_groups(model, backbone_name, phase2_lr_backbone, phase2_lr_head)
    optimizer = optim.AdamW(param_groups, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=phase2_epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    for epoch in range(1, phase2_epochs + 1):
        start_time = time.time()

        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler, use_amp)
        val_metrics = evaluate(model, val_loader, criterion, device)

        scheduler.step()
        lr_bb = optimizer.param_groups[0]["lr"]
        lr_hd = optimizer.param_groups[1]["lr"]
        epoch_time = time.time() - start_time

        global_epoch = phase1_epochs + epoch
        record = {
            "phase": 2,
            "epoch": global_epoch,
            "lr_backbone": lr_bb,
            "lr_head": lr_hd,
            "train_loss": train_metrics["loss"],
            "train_top1": train_metrics["top1"],
            "train_top5": train_metrics["top5"],
            "val_loss": val_metrics["loss"],
            "val_top1": val_metrics["top1"],
            "val_top5": val_metrics["top5"],
            "epoch_time_sec": epoch_time,
        }
        history.append(record)

        marker = ""
        if record["val_top1"] > best_val_top1:
            best_val_top1 = record["val_top1"]
            torch.save(
                {
                    "epoch": global_epoch,
                    "phase": 2,
                    "backbone": backbone_name,
                    "model_state_dict": model.state_dict(),
                    "best_val_top1": best_val_top1,
                    "history": history,
                    "class_to_idx": class_to_idx,
                },
                checkpoint_path,
            )
            marker = " ★"

        print(
            f"  P2 Epoch {epoch:02d}/{phase2_epochs} (global {global_epoch:02d}) | "
            f"lr_bb={lr_bb:.7f} lr_hd={lr_hd:.6f} | "
            f"train_top1={record['train_top1']:.4f} | "
            f"val_top1={record['val_top1']:.4f} | val_top5={record['val_top5']:.4f} | "
            f"{epoch_time:.1f}s{marker}"
        )

    history_path.write_text(json.dumps(history, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\n  Final best val_top1: {best_val_top1:.4f}")
    print(f"  Checkpoint: {checkpoint_path}")
    print(f"  History:    {history_path}")

    return model, history

## Huấn luyện cả 3 backbones

Chạy pipeline tuần tự cho từng backbone. Mỗi backbone sẽ:
1. Phase 1 — Train head (5 epochs)
2. Phase 2 — Fine-tune toàn mạng (15 epochs)

Kết quả (checkpoint, history JSON) được lưu riêng biệt theo tên backbone.

In [ ]:
all_results: dict[str, dict] = {}
all_histories: dict[str, list[dict]] = {}

for backbone_name in BACKBONE_REGISTRY:
    model, history = run_transfer_pipeline(
        backbone_name=backbone_name,
        num_classes=num_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        use_amp=USE_AMP,
        phase1_epochs=PHASE1_EPOCHS,
        phase2_epochs=PHASE2_EPOCHS,
        phase1_lr=PHASE1_LR,
        phase2_lr_backbone=PHASE2_LR_BACKBONE,
        phase2_lr_head=PHASE2_LR_HEAD,
        weight_decay=WEIGHT_DECAY,
        checkpoint_dir=CHECKPOINT_DIR,
        history_dir=HISTORY_DIR,
    )
    all_histories[backbone_name] = history

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\nAll backbones trained.")

## Training curves

Vẽ loss và accuracy curves cho cả 3 backbones trên cùng một figure để so sánh.
Đường kẻ dọc đánh dấu ranh giới giữa Phase 1 (feature extraction) và Phase 2 (fine-tuning).

In [ ]:
for name in BACKBONE_REGISTRY:
    if name not in all_histories:
        path = HISTORY_DIR / f"{name}_history.json"
        if path.exists():
            all_histories[name] = load_json(path)

COLORS = {"resnet50": "#1f77b4", "mobilenet_v2": "#ff7f0e", "efficientnet_b0": "#2ca02c"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for backbone_name, history in all_histories.items():
    epochs = [r["epoch"] for r in history]
    color = COLORS.get(backbone_name, None)

    axes[0].plot(epochs, [r["val_loss"] for r in history], label=f"{backbone_name} val", color=color)
    axes[0].plot(epochs, [r["train_loss"] for r in history], label=f"{backbone_name} train", color=color, linestyle="--", alpha=0.5)

    axes[1].plot(epochs, [r["val_top1"] for r in history], label=f"{backbone_name} val", color=color)
    axes[1].plot(epochs, [r["train_top1"] for r in history], label=f"{backbone_name} train", color=color, linestyle="--", alpha=0.5)

    axes[2].plot(epochs, [r["val_top5"] for r in history], label=f"{backbone_name} val", color=color)
    axes[2].plot(epochs, [r["train_top5"] for r in history], label=f"{backbone_name} train", color=color, linestyle="--", alpha=0.5)

for ax in axes:
    ax.axvline(x=PHASE1_EPOCHS, color="gray", linestyle=":", alpha=0.6, label="Phase boundary")
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curves")

axes[1].set_ylabel("Top-1 Accuracy")
axes[1].set_title("Top-1 Accuracy Curves")

axes[2].set_ylabel("Top-5 Accuracy")
axes[2].set_title("Top-5 Accuracy Curves")

plt.suptitle("Transfer Learning — Training Curves Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Đánh giá trên test set

Load lại best checkpoint của mỗi backbone và đánh giá trên tập test.
Tổng hợp kết quả vào bảng so sánh cuối cùng.

In [ ]:
criterion_eval = nn.CrossEntropyLoss()

test_results: dict[str, dict] = {}

for backbone_name in BACKBONE_REGISTRY:
    ckpt_path = CHECKPOINT_DIR / f"{backbone_name}_best.pt"
    if not ckpt_path.exists():
        print(f"Checkpoint not found for {backbone_name}, skipping.")
        continue

    checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    model = build_transfer_model(backbone_name, num_classes, dropout=0.3, freeze_backbone=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(DEVICE)

    test_metrics = evaluate(model, test_loader, criterion_eval, DEVICE)

    test_results[backbone_name] = {
        "best_epoch": checkpoint["epoch"],
        "best_phase": checkpoint["phase"],
        "val_top1": checkpoint["best_val_top1"],
        "test_loss": test_metrics["loss"],
        "test_top1": test_metrics["top1"],
        "test_top5": test_metrics["top5"],
        "total_params": count_total_parameters(model),
    }

    print(
        f"{backbone_name:20s} | "
        f"best epoch: {checkpoint['epoch']:2d} (P{checkpoint['phase']}) | "
        f"val_top1: {checkpoint['best_val_top1']:.4f} | "
        f"test_top1: {test_metrics['top1']:.4f} | "
        f"test_top5: {test_metrics['top5']:.4f}"
    )

    del model, checkpoint
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Bảng so sánh tổng hợp

So sánh kết quả của 3 backbone transfer learning với AlexNet from scratch (tuần 6-7).

In [ ]:
rows = []

alexnet_history_path = HISTORY_DIR / "alexnet_from_scratch_history.json"
if alexnet_history_path.exists():
    alexnet_history = load_json(alexnet_history_path)
    best_alexnet = max(alexnet_history, key=lambda r: r["val_top1"])

    rows.append({
        "Model": "AlexNet (scratch)",
        "Params": "57.5M",
        "Best Epoch": best_alexnet["epoch"],
        "Val Top-1": f"{best_alexnet['val_top1']:.4f}",
        "Test Top-1": "N/A (val only)",
        "Test Top-5": "N/A (val only)",
    })

for name, res in test_results.items():
    rows.append({
        "Model": f"{name} (transfer)",
        "Params": f"{res['total_params']/1e6:.1f}M",
        "Best Epoch": f"{res['best_epoch']} (P{res['best_phase']})",
        "Val Top-1": f"{res['val_top1']:.4f}",
        "Test Top-1": f"{res['test_top1']:.4f}",
        "Test Top-5": f"{res['test_top5']:.4f}",
    })

if not rows:
    print("No results to display. Train at least one backbone first.")
else:
    header = rows[0].keys()
    col_widths = {k: max(len(k), max(len(str(r[k])) for r in rows)) for k in header}
    sep = " | ".join("-" * col_widths[k] for k in header)
    header_str = " | ".join(k.ljust(col_widths[k]) for k in header)

    print(header_str)
    print(sep)
    for row in rows:
        print(" | ".join(str(row[k]).ljust(col_widths[k]) for k in header))

In [ ]:
if test_results:
    names = list(test_results.keys())
    top1_scores = [test_results[n]["test_top1"] * 100 for n in names]
    top5_scores = [test_results[n]["test_top5"] * 100 for n in names]

    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - width / 2, top1_scores, width, label="Test Top-1 (%)", color="#4c72b0")
    bars2 = ax.bar(x + width / 2, top5_scores, width, label="Test Top-5 (%)", color="#55a868")

    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Transfer Learning — Test Accuracy Comparison")
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.legend()
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.3)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()

## Visualize predictions

Chọn model có **val_top1** cao nhất (tránh leak test set vào model selection), sau đó hiển thị dự đoán trên test set.

In [ ]:
def denormalize_image(image_tensor: torch.Tensor) -> np.ndarray:
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

    image = image_tensor.detach().cpu() * std + mean
    image = image.clamp(0.0, 1.0)
    return image.permute(1, 2, 0).numpy()


@torch.inference_mode()
def visualize_predictions(
    model: nn.Module,
    dataset: Dataset,
    idx_to_class: dict[int, str],
    device: torch.device,
    title: str = "",
    num_samples: int = 8,
) -> None:
    model.eval()

    chosen_indices = random.sample(range(len(dataset)), k=min(num_samples, len(dataset)))
    cols = 4
    rows = int(np.ceil(len(chosen_indices) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 4.2 * rows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, index in zip(axes, chosen_indices):
        image_tensor, label = dataset[index]
        logits = model(image_tensor.unsqueeze(0).to(device))
        prediction = int(logits.argmax(dim=1).item())

        true_name = idx_to_class[label].split("-", 1)[1].replace("_", " ")
        pred_name = idx_to_class[prediction].split("-", 1)[1].replace("_", " ")
        correct = prediction == label

        ax.imshow(denormalize_image(image_tensor))
        ax.set_title(
            f"True: {true_name}\nPred: {pred_name}",
            color="green" if correct else "red",
            fontsize=10,
        )

    if title:
        plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


if test_results:
    best_backbone = max(test_results, key=lambda n: test_results[n]["val_top1"])
    ckpt_path = CHECKPOINT_DIR / f"{best_backbone}_best.pt"
    checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    best_model = build_transfer_model(best_backbone, num_classes, dropout=0.3, freeze_backbone=False)
    best_model.load_state_dict(checkpoint["model_state_dict"])
    best_model = best_model.to(DEVICE)

    res = test_results[best_backbone]
    visualize_predictions(
        best_model,
        test_dataset,
        idx_to_class,
        DEVICE,
        title=f"Best model (by val): {best_backbone} | val_top1={res['val_top1']:.4f}, test_top1={res['test_top1']:.4f}",
        num_samples=12,
    )

    del best_model, checkpoint
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Nhận xét và gợi ý bước tiếp theo

**Kỳ vọng kết quả:**
- AlexNet from scratch trên Stanford Dogs (120 classes, ~14K train) thường đạt ~20-30% test top-1
- Transfer learning từ ImageNet pretrained nên tăng đáng kể:
  - **ResNet50**: ~75-85% test top-1 (backbone mạnh, nhiều params)
  - **MobileNetV2**: ~70-80% (lightweight nhưng hiệu quả)
  - **EfficientNet-B0**: ~75-85% (compound scaling tối ưu)

**Gợi ý tiếp theo:**
- Thử `EfficientNet-B2` hoặc `B4` nếu GPU cho phép (lớn hơn → accuracy cao hơn)
- Thêm `CutMix` / `MixUp` augmentation cho Phase 2
- Tăng `PHASE2_EPOCHS` lên 25-30 nếu validation accuracy vẫn đang tăng
- Thêm `classification report` / `confusion matrix` để phân tích lỗi chi tiết
- So sánh với `ViT-B/16` (Vision Transformer) cho tuần tiếp theo